# Variational Autoencoder (VAE) für Synthetische Daten
Reconstructed Notebook

In [1]:

import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import pickle # Zur Persistierung

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [2]:

# ------------------------- Daten laden -------------------------
from pathlib import Path
current_dir = Path.cwd()
notebooks_root = current_dir.parents[2]
if not notebooks_root.name == "Jupyter Notebooks":
    for p in current_dir.parents:
        if p.name == "Jupyter Notebooks":
            notebooks_root = p
            break

preprocessing_dir = notebooks_root / "3_Machine-Learning" / "3.1_Preprocessing" / "Preprocessing"
print(f"Preprocessing Verzeichnis: {preprocessing_dir}")

if not preprocessing_dir.exists():
    # Fallback auf relativ, falls absolut fehlschlägt
    preprocessing_dir = Path("../../../3_Machine-Learning/3.1_Preprocessing/Preprocessing")

if not preprocessing_dir.exists():
     raise FileNotFoundError(f"Verzeichnis existiert nicht: {preprocessing_dir}")

all_subdirs = [d for d in preprocessing_dir.iterdir() if d.is_dir()]
if not all_subdirs:
    raise FileNotFoundError("Keine Preprocessing-Ordner gefunden.")

latest_subdir = max(all_subdirs, key=os.path.getmtime)
data_path = latest_subdir / 'Preprocessed_SOM_Ready.csv'

print(f"Lade Daten aus: {data_path}")

try:
    df = pd.read_csv(data_path, low_memory=False)
    print(f"Datensatzgröße: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Fehler beim Laden: {e}")
    raise e


# ----------------- Filter Unknown Rock Types -----------------
# Anforderung: Keine unbekannten Gesteinsarten trainieren.
if 'rock_type' in df.columns:
    initial_len = len(df)
    df['rock_type'] = df['rock_type'].astype(str)
    
    # Filter
    df = df[df['rock_type'] != 'Unknown'].copy()
    df = df[df['rock_type'] != 'nan'].copy()
    
    filtered_len = len(df)
    print(f"Filtered 'Unknown' Rock Types: {initial_len} -> {filtered_len} rows (Removed {initial_len - filtered_len})")
else:
    print("Warning: 'rock_type' column not found for filtering.")


Preprocessing Verzeichnis: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing
Lade Daten aus: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-01-05_00-04-27\Preprocessed_SOM_Ready.csv


Datensatzgröße: (94264, 24)


,WGS84_latitude,WGS84_longitude,Database_number,temperature_in_c,rock_type,redox_potential_in_mV_gauss,total_dissolved_solids_in_mmol/L_log_gauss,O2_in_mmol/L_log_gauss,Na_in_mmol/L_log_gauss,Mg_in_mmol/L_log_gauss,...,Li_in_mmol/L_log_gauss,K_in_mmol/L_log_gauss,Sr_in_umol/L_log_gauss,NH4_in_umol/L_log_gauss,Fe_in_mmol/L_log_gauss,Mn_in_mmol/L_log_gauss,F_in_umol/L_log_gauss,NO3_in_mmol/L_log_gauss,H2SiO3_in_umol/L_log_gauss,HS_in_mmol/L_log_gauss
0,46.19680,8.54160,5,25.7,Andesite,NaN,-2.407685,NaN,-2.002380,-2.696511,...,-1.767903,-2.225823,NaN,NaN,NaN,NaN,0.590949,NaN,NaN,NaN
1,46.24352,9.72534,5,37.9,Basalt,NaN,-2.153160,NaN,-1.875864,-2.823144,...,-1.634747,-1.625274,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,45.76189,6.96890,5,22.3,Andesite,NaN,-1.933503,NaN,-2.035320,-0.746430,...,-1.404533,-1.674186,NaN,NaN,NaN,NaN,-0.926176,NaN,NaN,NaN
3,46.26274,8.10919,5,29.2,Andesite,NaN,-1.847320,NaN,-3.276940,-0.702315,...,NaN,-1.837975,NaN,NaN,NaN,NaN,-0.926176,NaN,NaN,NaN
4,46.26274,8.10919,5,38.6,Andesite,NaN,-1.732235,NaN,-2.709462,-0.653377,...,NaN,-1.775299,NaN,NaN,NaN,NaN,-0.926176,NaN,NaN,NaN


Filtered 'Unknown' Rock Types: 94264 -> 24332 rows (Removed 69932)


In [3]:

# ------------------------- Data Preparation & Feature Engineering -------------------------

# 1. Temperatur-Behandlung (Vor Dropping schützen)
if 'temperature_in_c' in df.columns:
    df['temperature_in_c'] = pd.to_numeric(df['temperature_in_c'], errors='coerce')
    temp_median = df['temperature_in_c'].median()
    if pd.isna(temp_median): temp_median = df['temperature_in_c'].mean() 
    if pd.isna(temp_median): temp_median = 0 
    df['temperature_in_c'] = df['temperature_in_c'].fillna(temp_median)
    print(f"Temperature filled with: {temp_median}")
else:
    print("WARNING: 'temperature_in_c' not found!")

# 2. Rock Type Handling (One-Hot Encoding)
if 'rock_type' in df.columns:
    df['rock_type'] = df['rock_type'].fillna('Unknown')
    rock_dummies = pd.get_dummies(df['rock_type'], prefix='rock', dtype=float)
    df_clean = pd.concat([df, rock_dummies], axis=1)
    df_clean.drop(columns=['rock_type'], inplace=True)
    rock_cols = rock_dummies.columns.tolist()
    print(f"Added {len(rock_cols)} rock type columns.")
else:
    print("WARNING: 'rock_type' not found!")
    df_clean = df.copy()
    rock_cols = []

# 3. Filter Columns (>95% Missing)
drop_threshold = 95.0
missing_percentage = df_clean.isnull().mean() * 100
cols_to_drop = missing_percentage[missing_percentage > drop_threshold].index.tolist()
protected_cols = ['temperature_in_c'] + rock_cols
cols_to_drop = [c for c in cols_to_drop if c not in protected_cols]

if cols_to_drop:
    print(f"Dropping: {cols_to_drop}")
    df_clean = df_clean.drop(columns=cols_to_drop)
else:
    print("No columns dropped.")

# 4. Exclude Metadata
metadata_cols = ['Database_number', 'database_name', 'Database_name', 'Date', 'Day', 'Month', 'Year', 'WGS84_latitude', 'WGS84_longitude']
cols_to_exclude = [c for c in metadata_cols if c in df_clean.columns]
if cols_to_exclude:
    df_train = df_clean.drop(columns=cols_to_exclude)
else:
    df_train = df_clean

# 5. Numeric Prep
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
for c in numeric_cols:
    df_train[c] = pd.to_numeric(df_train[c], errors='coerce')
data_values = df_train[numeric_cols].values

# 6. Imputation & Scaling
print("Imputing & Scaling...")
imputer = SimpleImputer(strategy='mean')
data_imputed = imputer.fit_transform(data_values)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_imputed)

# Feature-Namen explizit definieren
feature_names = numeric_cols
print(f"Final Features ({len(feature_names)}): {feature_names}")

# 7. Persistierung (Metadaten speichern, um NameError zu vermeiden)
metadata = {
    'feature_names': feature_names,
    'rock_cols': rock_cols,
    'scaler': scaler,
    'imputer': imputer
}
with open('vae_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("Metadaten gespeichert unter vae_metadata.pkl")

# 8. Tensors
tensor_data = torch.FloatTensor(data_scaled)
batch_size = 64
dataset = TensorDataset(tensor_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


Temperature filled with: 24.45
Added 30 rock type columns.
Dropping: ['redox_potential_in_mV_gauss', 'O2_in_mmol/L_log_gauss', 'NH4_in_umol/L_log_gauss', 'Mn_in_mmol/L_log_gauss', 'F_in_umol/L_log_gauss', 'NO3_in_mmol/L_log_gauss', 'H2SiO3_in_umol/L_log_gauss', 'HS_in_mmol/L_log_gauss']
Imputing & Scaling...
Final Features (42): ['temperature_in_c', 'total_dissolved_solids_in_mmol/L_log_gauss', 'Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L_log_gauss', 'Li_in_mmol/L_log_gauss', 'K_in_mmol/L_log_gauss', 'Sr_in_umol/L_log_gauss', 'Fe_in_mmol/L_log_gauss', 'rock_Andesite', 'rock_Anhydrite', 'rock_Basalt', 'rock_Breccia', 'rock_Chat', 'rock_Chert', 'rock_Claystone', 'rock_Coal', 'rock_Conglomerate', 'rock_Dolerite', 'rock_Dolomite', 'rock_Gneiss', 'rock_Granite', 'rock_Graywacke', 'rock_Greenschist', 'rock_Gypsum', 'rock_Limestone', 'rock_Marble', 'rock_Marl', 'rock_Metamorphic Rock', 'rock_

In [4]:

# ------------------------- VAE Model -------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=10):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
        self.activation = nn.ReLU()

    def encode(self, x):
        h1 = self.activation(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = self.activation(self.fc3(z))
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# Initialisierung
input_dim = data_scaled.shape[1]
latent_dim = 8
model = VAE(input_dim, hidden_dim=64, latent_dim=latent_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print("Model initialized.")


Model initialized.


In [5]:

# ------------------------- Training -------------------------
def loss_function(recon_x, x, mu, logvar):
    # MSE Loss für Rekonstruktion
    MSE = nn.MSELoss(reduction='sum')(recon_x, x)
    # KL Divergenz
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return MSE + KLD

epochs = 50
model.train()
print("Starting training...")
for epoch in range(epochs):
    train_loss = 0
    for batch_idx, (data,) in enumerate(dataloader):
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch} Average loss: {train_loss / len(dataloader.dataset):.4f}')

print("Training finished.")


Starting training...


Epoch: 0 Average loss: 38.3330


Epoch: 10 Average loss: 17.8296


Epoch: 20 Average loss: 14.9755


Epoch: 30 Average loss: 13.6510


Epoch: 40 Average loss: 12.7585


Training finished.


In [6]:

# ------------------------- Synthetische Datengenerierung & Speichern -------------------------
print("Generiere synthetische Daten...")

# Metadaten laden, um Variablen sicherzustellen
try:
    with open('vae_metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
        feature_names = metadata['feature_names']
        rock_cols = metadata['rock_cols']
        scaler = metadata['scaler'] # Sicherstellen, dass der gefittete Scaler verwendet wird.
    print("Metadaten erfolgreich geladen.")
except Exception as e:
    print(f"Could not load metadata: {e}")
    # Falls Variablen bereits im Scope existieren, diese nutzen. 
    # Andernfalls Fehler werfen.
    if 'feature_names' not in locals():
        raise NameError("feature_names not defined and metadata load failed!")

model.eval()
num_samples = len(df)
with torch.no_grad():
    z_sample = torch.randn(num_samples, latent_dim)
    generated_data_scaled = model.decode(z_sample).numpy()

# Inverse Transformation
print("Inverse transforming scaling...")
generated_data_imputed = scaler.inverse_transform(generated_data_scaled)
df_synthetic = pd.DataFrame(generated_data_imputed, columns=feature_names)

# Inverses OHE
if rock_cols:
    print("Reconstructing Rock Types...")
    rock_data = df_synthetic[rock_cols]
    rock_types = rock_data.idxmax(axis=1).apply(lambda x: x.replace('rock_', ''))
    df_synthetic['rock_type'] = rock_types
    df_synthetic.drop(columns=rock_cols, inplace=True)

# Koordinaten
print("Adding coordinates...")
coords = df[['WGS84_latitude', 'WGS84_longitude']].sample(n=num_samples, replace=True).reset_index(drop=True)
df_synthetic = pd.concat([df_synthetic, coords], axis=1)

# Speichern
output_filename = "Synthetic_Hydrogeochemistry_Data_Imputed.csv"
df_synthetic.to_csv(output_filename, index=False)
print(f"Gespeichert unter: {output_filename}")
print(f"Columns: {df_synthetic.columns.tolist()}")
display(df_synthetic.head())


Generiere synthetische Daten...
Metadaten erfolgreich geladen.
Inverse transforming scaling...
Reconstructing Rock Types...
Adding coordinates...


Gespeichert unter: Synthetic_Hydrogeochemistry_Data_Imputed.csv
Columns: ['temperature_in_c', 'total_dissolved_solids_in_mmol/L_log_gauss', 'Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L_log_gauss', 'Li_in_mmol/L_log_gauss', 'K_in_mmol/L_log_gauss', 'Sr_in_umol/L_log_gauss', 'Fe_in_mmol/L_log_gauss', 'rock_type', 'WGS84_latitude', 'WGS84_longitude']


,temperature_in_c,total_dissolved_solids_in_mmol/L_log_gauss,Na_in_mmol/L_log_gauss,Mg_in_mmol/L_log_gauss,Ca_in_mmol/L_log_gauss,Cl_in_mmol/L_log_gauss,SO4_in_mmol/L_log_gauss,HCO3_in_mmol/L_log_gauss,Li_in_mmol/L_log_gauss,K_in_mmol/L_log_gauss,Sr_in_umol/L_log_gauss,Fe_in_mmol/L_log_gauss,rock_type,WGS84_latitude,WGS84_longitude
0,24.260046,-0.731080,-0.784874,-0.592878,-0.685448,-0.720447,-0.354144,0.470896,0.068632,-0.480224,0.095217,0.149266,Limestone,31.30180,-90.37010
1,23.332499,-1.108051,-1.065570,-1.476564,-1.598979,-1.145530,-0.617396,1.353626,0.080479,-0.590593,0.116485,0.176607,Sandstone,44.25752,-105.20422
2,22.856789,-0.648404,-0.583022,-1.207153,-1.343841,-0.615957,-1.255548,1.016557,0.094938,-0.640033,0.092693,0.178980,Sandstone,26.24600,-98.62700
3,24.078669,-1.084176,-1.091361,-0.684462,-0.747292,-1.141849,1.103123,0.592938,0.148162,-0.152417,0.152896,0.255690,Sandstone,41.56068,-108.42234
4,24.211563,0.393699,0.387037,0.606713,0.502840,0.388525,0.699705,-0.207580,0.120844,-0.019469,0.206310,0.209507,Sandstone,33.56940,-102.44380
